# Telecom Customer Churn Prediction
## Sprint 1: Exploratory Data Analysis | Sprint 2: Model Building
**Domain:** Telecommunication | **Problem:** Churn Prediction | **Task:** Binary Classification

---
## Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')

# Color palette
BLUE   = '#2E4770'
ORANGE = '#E07B39'
TEAL   = '#3A9EA5'
GREEN  = '#27AE60'
RED    = '#C0392B'
PALETTE = [BLUE, ORANGE, TEAL, GREEN, RED, '#9B59B6', '#F39C12', '#B0B8C1']

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.edgecolor':   '#CCCCCC',
    'axes.grid':        True,
    'grid.color':       '#EEEEEE',
    'grid.linewidth':   0.7,
    'axes.titlesize':   12,
    'axes.labelsize':   10,
})

print('Libraries imported successfully.')

---
## Step 1 — Load Dataset

In [ ]:
df = pd.read_csv('churn_dataset.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
df.info()

In [ ]:
# Fix TotalCharges — contains blank spaces, convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Null values before drop: {df.isnull().sum().sum()}')

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Shape after cleaning: {df.shape}')
print(f'Null values after drop: {df.isnull().sum().sum()}')

In [ ]:
df.describe()

---
# SPRINT 1 — Exploratory Data Analysis (EDA)

### 1.1 — Target Variable: Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print(f'Churn Counts:\n{churn_counts}')
print(f'\nChurn Rate: {churn_pct["Yes"]:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Sprint 1 — Churn Distribution Overview', fontsize=14, fontweight='bold', color=BLUE)

# Bar chart
axes[0].bar(['No Churn', 'Churned'], churn_counts.values, color=[BLUE, ORANGE], edgecolor='white', width=0.5)
axes[0].set_title('Customer Churn Count')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churned'],
            autopct='%1.1f%%', colors=[BLUE, ORANGE],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Percentage')

plt.tight_layout()
plt.show()

### 1.2 — Numerical Variable Distributions vs Churn

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Sprint 1 — Numerical Variables vs Churn', fontsize=14, fontweight='bold', color=BLUE)

for i, col in enumerate(num_cols):
    # Histogram
    for label, color in zip(['No', 'Yes'], [BLUE, ORANGE]):
        axes[0, i].hist(df[df['Churn'] == label][col], bins=30,
                        alpha=0.6, color=color, label=label, edgecolor='white')
    axes[0, i].set_title(f'{col} — Distribution')
    axes[0, i].set_xlabel(col)
    axes[0, i].legend(title='Churn')

    # Boxplot
    data_plot = [df[df['Churn'] == 'No'][col].values,
                 df[df['Churn'] == 'Yes'][col].values]
    bp = axes[1, i].boxplot(data_plot, patch_artist=True,
                             labels=['No Churn', 'Churned'],
                             medianprops={'color': 'white', 'linewidth': 2})
    bp['boxes'][0].set_facecolor(BLUE)
    bp['boxes'][1].set_facecolor(ORANGE)
    axes[1, i].set_title(f'{col} — Boxplot by Churn')
    axes[1, i].set_ylabel(col)

plt.tight_layout()
plt.show()

### 1.3 — Categorical Variables vs Churn Rate

In [ ]:
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'InternetService', 'Contract',
            'PaymentMethod', 'PaperlessBilling']

avg_churn = churn_pct['Yes']

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle('Sprint 1 — Categorical Variables vs Churn Rate', fontsize=14, fontweight='bold', color=BLUE)
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    cr = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
    cr.columns = [col, 'ChurnRate']
    cr = cr.sort_values('ChurnRate', ascending=False)

    bars = axes[i].bar(cr[col].astype(str), cr['ChurnRate'],
                       color=PALETTE[:len(cr)], edgecolor='white')
    axes[i].set_title(f'{col} vs Churn Rate')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_xticklabels(cr[col].astype(str), rotation=20, ha='right', fontsize=8)
    axes[i].axhline(y=avg_churn, color=RED, linestyle='--', linewidth=1,
                    label=f'Avg {avg_churn:.1f}%')
    axes[i].legend(fontsize=8)
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                     f"{bar.get_height():.1f}%", ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

### 1.4 — Correlation Heatmap

In [ ]:
df_corr = df.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df_corr[corr_cols].corr(), annot=True, fmt='.2f',
            cmap='Blues', linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Sprint 1 — Correlation Heatmap', fontweight='bold', color=BLUE)
plt.tight_layout()
plt.show()

### 1.5 — Churn Rate by Tenure Group

In [ ]:
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],
                             labels=['0-12 mo', '13-24 mo', '25-48 mo', '49-72 mo'])

tg = df.groupby('tenure_group', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100).reset_index()
tg.columns = ['Tenure Group', 'ChurnRate']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tg['Tenure Group'].astype(str), tg['ChurnRate'],
              color=[ORANGE, TEAL, GREEN, BLUE], edgecolor='white', width=0.5)
ax.set_title('Sprint 1 — Churn Rate by Tenure Group', fontweight='bold', color=BLUE)
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Tenure Group')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 1.6 — Key Insights & Business Recommendations

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean()*100).round(1)
internet_churn = df.groupby('InternetService')['Churn'].apply(lambda x: (x=='Yes').mean()*100).round(1)
senior_churn   = df.groupby('SeniorCitizen')['Churn'].apply(lambda x: (x=='Yes').mean()*100).round(1)

print('='*55)
print('SPRINT 1 — KEY INSIGHTS')
print('='*55)
print(f'Insight 1: Overall churn rate = {churn_pct["Yes"]:.1f}% (class imbalance noted)')
print(f'\nInsight 2: Churn by Contract type:')
print(contract_churn.to_string())
print(f'\nInsight 3: Churn by Internet Service:')
print(internet_churn.to_string())
print(f'\nInsight 4: Senior Citizen churn = {senior_churn.get(1,0):.1f}% vs Non-Senior = {senior_churn.get(0,0):.1f}%')
print(f'Insight 5: 0-12 month tenure customers churn at {tg.iloc[0]["ChurnRate"]:.1f}% — highest risk group')
print(f'Insight 6: MonthlyCharges positively correlated with churn')

print('\n' + '='*55)
print('BUSINESS RECOMMENDATIONS')
print('='*55)
print('R1: Incentivize month-to-month customers to upgrade to annual contracts')
print('R2: Launch 90-day onboarding loyalty program for new customers (0-12 mo)')
print('R3: Bundle security/support add-ons for Fiber Optic users at discounted rate')
print('R4: Design senior citizen retention packages with simplified billing')
print('R5: Flag customers with high monthly charges + low tenure as high-risk')

---
# SPRINT 2 — Data Preparation & Model Building

### Step 2 — Identify Variables & Task Type

In [ ]:
print('Input Variables (X) : All columns except customerID and Churn')
print('Target Variable (y) : Churn  (Yes = 1, No = 0)')
print('ML Task Type        : Binary Classification')
print('Evaluation Metric   : Accuracy Score')
print('                      metrics.accuracy_score(actual, predicted)')

### Step 3 — Train / Test Split (75:25)

In [ ]:
drop_cols = ['customerID', 'Churn', 'tenure_group']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = (df['Churn'] == 'Yes').astype(int)

# Identify feature types
cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Categorical features ({len(cat_features)}): {cat_features}')
print(f'Numerical features  ({len(num_features)}): {num_features}')

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f'\nTrain set: {X_train.shape}')
print(f'Test set : {X_test.shape}')

### Step 4 — Data Preparation on Train Data
- **Categorical (binary)** → Label Encoding  
- **Categorical (multi-class)** → One Hot Encoding  
- **Numerical** → StandardScaler (Fit + Transform on train)

In [ ]:
binary_cats = [c for c in cat_features if X[c].nunique() == 2]
multi_cats  = [c for c in cat_features if X[c].nunique() > 2]

print(f'Label Encoding (binary 2-class): {binary_cats}')
print(f'OneHot Encoding (multi-class)  : {multi_cats}')

# Label Encoding on binary categoricals
le = LabelEncoder()
for col in binary_cats:
    X_train[col] = le.fit_transform(X_train[col])

# One Hot Encoding on multi-class categoricals
X_train = pd.get_dummies(X_train, columns=multi_cats, drop_first=True)

# Standardization on numerical features
scaler = StandardScaler()
num_cols_model = [c for c in num_features if c in X_train.columns]
X_train[num_cols_model] = scaler.fit_transform(X_train[num_cols_model])

print(f'\nFinal train feature count: {X_train.shape[1]}')
print('Standardization: FIT + TRANSFORM applied on train data')
X_train.head()

### Step 5 — Data Preparation on Test Data
- Same encoding applied (Transform only — NO fit on test)

In [ ]:
# Label Encoding on test
for col in binary_cats:
    X_test[col] = le.fit_transform(X_test[col])

# One Hot Encoding on test
X_test = pd.get_dummies(X_test, columns=multi_cats, drop_first=True)

# Align columns (in case OHE creates different columns)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Standardization — TRANSFORM only (scaler already fit on train)
X_test[num_cols_model] = scaler.transform(X_test[num_cols_model])

print(f'Test feature count : {X_test.shape[1]}')
print('Standardization: TRANSFORM only applied on test data (no fit)')
X_test.head()

### Step 6 — Model Training
Training 5 algorithms: KNN, Logistic Regression, SVM, Decision Tree, Random Forest

In [ ]:
models = {
    'KNN'                : KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM'                : SVC(kernel='rbf', random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'{name} — trained.')

### Step 7 — Predict & Evaluate Each Model

In [ ]:
results = {}

print('-' * 45)
print(f'{"Algorithm":<22} {"Accuracy":>10}')
print('-' * 45)

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = metrics.accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'{name:<22} {acc*100:>9.2f}%')

print('-' * 45)
best_model = max(results, key=results.get)
print(f'\nBest Algorithm: {best_model} ({results[best_model]*100:.2f}%)')

### Step 8 — Algorithm Comparison Plot

In [ ]:
names  = list(results.keys())
accs   = [v * 100 for v in results.values()]
colors = [GREEN if n == best_model else BLUE for n in names]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(names, accs, color=colors, edgecolor='white', width=0.55)

ax.set_title('Sprint 2 — Algorithm Accuracy Comparison\nTelecom Customer Churn Prediction',
             fontweight='bold', color=BLUE, fontsize=13)
ax.set_ylabel('Accuracy (%)')
ax.set_xlabel('Algorithm')
ax.set_ylim([min(accs) - 3, 100])
ax.axhline(y=max(accs), color=RED, linestyle='--', linewidth=1.2,
           label=f'Best: {max(accs):.2f}%')
ax.legend(fontsize=10)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=10)

# Highlight best
best_idx = names.index(best_model)
bars[best_idx].set_edgecolor(GREEN)
bars[best_idx].set_linewidth(2.5)

plt.tight_layout()
plt.show()

print(f'\nConclusion: {best_model} achieved the highest accuracy of {results[best_model]*100:.2f}%')
print('It is the recommended model for Telecom Churn Prediction.')

### Detailed Report — Best Model

In [ ]:
best = models[best_model]
y_pred_best = best.predict(X_test)

print(f'Classification Report: {best_model}')
print(metrics.classification_report(y_test, y_pred_best, target_names=['No Churn', 'Churned']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = metrics.confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churned'],
            yticklabels=['No Churn', 'Churned'], ax=ax)
ax.set_title(f'Confusion Matrix — {best_model}', fontweight='bold', color=BLUE)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## Final Conclusion

**Sprint 1 — EDA:**
- Overall churn rate is **26.6%** — class imbalance is present.
- **Month-to-month contract** customers churn at 42.7% vs 2.8% for 2-year contracts.
- **Fiber Optic** internet users churn at 41.9% — significantly above average.
- **New customers (0–12 months)** are the highest-risk segment at 47.7% churn.
- **Senior citizens** churn at 41.7% vs 23.7% for non-seniors.
- Monthly charges are positively correlated with churn probability.

**Sprint 2 — Model Building:**
- 5 algorithms evaluated on a 75:25 stratified train-test split.
- **Logistic Regression** achieved the best accuracy of **80.60%**.
- Recommended for production use: interpretable, fast, and accurate.
- Business value: Early identification of at-risk customers enables targeted retention campaigns, reducing churn and improving customer lifetime value (LTV).